# Preprocessing — krok 1: konsolidacja obserwacji

Cel: z wielu obserwacji tej samej pary (kurs, przystanek), zapisywanych co 15 s,
wybrać jedną, ostateczną wartość.
Praca nadal na próbce 4 dni (22, 23, 25, 26 października), feed A.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from config import RAW
from src.analysis.consolidate import consolidate_trip_updates

skonsolidowane = consolidate_trip_updates(RAW, feed="A")
print("Liczba wierszy po konsolidacji:", f"{len(skonsolidowane):,}")
skonsolidowane.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Liczba wierszy po konsolidacji: 686,871


,date,trip_id,stop_id,route_id,stop_sequence,arrival_delay,departure_delay,liczba_obserwacji,ostatnia_obserwacja
0,2026-10-22,20261021_1_129496544_8,13973,601,NaN,0.0,0.0,99,1792628880
1,2026-10-22,20261021_1_129496606_7,18288,608,NaN,0.0,0.0,31,1792627712
2,2026-10-22,20261021_1_129496662_4,7891,662,NaN,0.0,0.0,196,1792630550
3,2026-10-22,20261021_1_129496589_6,14166,608,NaN,0.0,0.0,26,1792627630
4,2026-10-22,20261021_1_129496589_6,10559,608,NaN,0.0,0.0,49,1792628016


In [2]:
import duckdb
from src.analysis.io import trip_updates_glob, READ_PARQUET_OPTS

zapytanie = f"""
WITH pary AS (
    SELECT
        date, trip_id, stop_id,
        count(*) AS n_total,
        count(stop_sequence) AS n_z_sequence
    FROM read_parquet('{trip_updates_glob(RAW)}', {READ_PARQUET_OPTS})
    WHERE feed = 'A'
    GROUP BY date, trip_id, stop_id
)
SELECT
    count(*) AS wszystkie_pary,
    sum(CASE WHEN n_z_sequence > 0 AND n_z_sequence < n_total THEN 1 ELSE 0 END) AS pary_niespojne
FROM pary
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 wszystkie_pary  pary_niespojne
         686871          3034.0


In [3]:
zapytanie_kontrolne = f"""
SELECT count(*) AS n
FROM (
    SELECT DISTINCT date, trip_id, stop_id
    FROM read_parquet('{trip_updates_glob(RAW)}', {READ_PARQUET_OPTS})
    WHERE feed = 'A'
)
"""
liczba_unikalnych_par = duckdb.query(zapytanie_kontrolne).to_df()["n"].iloc[0]

assert len(skonsolidowane) == liczba_unikalnych_par, (
    f"Niezgodność: konsolidacja dała {len(skonsolidowane)} wierszy, "
    f"a unikalnych par jest {liczba_unikalnych_par}"
)
print(f"OK: {len(skonsolidowane):,} wierszy = {liczba_unikalnych_par:,} unikalnych par")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OK: 686,871 wierszy = 686,871 unikalnych par


## Test spójności i poprawka

Pierwsza wersja funkcji grupowała dane po `(date, trip_id, stop_id, stop_sequence)`.
Oparto to na założeniu, że `stop_sequence` jest spójne w obrębie pary (kurs, przystanek) we wszystkich
jej obserwacjach. **Założenie okazało się błędne** dla 3034 z 686871 par (~0.44%).
Pole bywa puste w części obserwacji tej samej pary, a wypełnione w innych.

Błąd skutkował tym, że takie pary były sztucznie dzielone na dwie grupy, każda z niepełnym
`arg_max`. Naniesiono poprawkę: `stop_sequence` zostało przeniesione
z klucza grupowania do agregacji, tak jak opóźnienia.

Po poprawce liczba wierszy po konsolidacji = liczba unikalnych par źródłowych
(686 871) - zweryfikowane asercją.